In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from b_Closed_form import *
from c_MC import *
from d_FDM import *
from e_1_run_cvae import * 
from e_2_CVAE import *
from generate import *



# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'barr' # van or barr
model_type = 'hes' # hes or bs

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

# cvae training settings
# _basic(eta에 B가 없는 버전), _B
BS_DATA_PATH = "/mnt/d/bs_dataset_basic.h5" 
BS_ETA  = "/mnt/d/bs_eta_basic.h5" 
HES_DATA_PATH = "/mnt/d/heston_dataset_basic.h5"
HES_ETA  = "/mnt/d/heston_eta_basic.h5" 

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942


/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
# 1. closed-form pricing result
closed_bs_van = BS_vanilla(BS_eta, opt_type)
closed_bs_barr = BS_barrier(BS_eta, B, opt_type)

In [ ]:
# 2. MC
mc_bs_van_gpu = MC_BS_vanilla_gpu(BS_eta, n_paths=100000, type=opt_type)
mc_hes_van_gpu = MC_heston_vanilla_gpu(Hes_eta, n_paths=100000, type=opt_type)
# BS_van
# CPU vanilla(1k, 10k, 100k) = (0.042, 0.453, 4.783)
# GPU vanilla(1k, 10k, 100k) = (0.143, 0.156, 0.163)

# Hes_van
# CPU vanilla(1k, 10k, 100k) = (0.112, 1.160, 11.992)
# GPU vanilla(1k, 10k, 100k) = (0.438, 0.485, 0.54)

#start = time.time()
mc_bs_barr_gpu = MC_BS_barrier_gpu(BS_eta, B, n_paths=100000, type=opt_type)
mc_hes_barr_gpu = MC_heston_barrier_gpu(Hes_eta, B, n_paths=100000, type=opt_type)
#print(f"{time.time() - start:.3f}s")
# BS_barr
# CPU vanilla(1k, 10k, 100k) = ( , , )
# GPU vanilla(1k, 10k, 100k) = ( 0.150, 0.150, 0.155)

# Hes_barr
# CPU vanilla(1k, 10k, 100k) = ( , , )
# GPU vanilla(1k, 10k, 100k) = ( 0.45, 0.47, 0.53)

In [ ]:
# 3. FDM
# vanilla
#ftcs_bs_van = FTCS_BS_vanilla(BS_eta, opt_type)
cn_bs_van = CN_BS_vanilla(BS_eta, opt_type)
cs_hes_van = CS_ADI_heston_vanilla(Hes_eta, opt_type)
#cs_hes_van = CS_ADI_heston_vanilla(Hes_eta, opt_type, dS=0.01, dv=0.0001, dt=0.01) # put인경우

# barrier
#ftcs_bs_barr = FTCS_BS_barrier(BS_eta, opt_type, B)
cn_bs_barr = CN_BS_barrier(BS_eta, opt_type, B)
cs_hes_barr = CS_ADI_heston_barrier(Hes_eta, opt_type, B)
#cs_hes_barr = CS_ADI_heston_barrier(Hes_eta, opt_type, B, dS=0.01, dv=0.0001, dt=0.01) # put인경우

In [ ]:
# 4. Gerate dataset for CVAE
# step1-1 BS
BS_paras = generate_BS_params(n_sets=100*(2**16), seed=1234)
bs_eta_dim = BS_paras.shape[1]

with h5py.File(BS_ETA, "w") as f:
    f.create_dataset("etas", data=BS_paras,
                    maxshape=(None, bs_eta_dim), 
                    chunks=(10240, bs_eta_dim), 
                    compression="gzip"
                    )

In [ ]:
# step2
# VANILLA X_T만 / Barrier X_T, M_T
chunk_dir  = "/mnt/d/bs_chunks_correction/"
chunk_size = 2**16 # 67M행 = 약 1GB
generate_dataset(BS_ETA, chunk_dir, 
                 model_type='bs', S0=S0, B=B, 
                 chunk_size=chunk_size
                 ) # 1ROUND : 5시간

round:0
[20000/655360] fail: 5264
[40000/655360] fail: 10488
[60000/655360] fail: 15839
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_000.h5 (67,108,864행)
[80000/655360] fail: 21112
[100000/655360] fail: 26353
[120000/655360] fail: 31804
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_001.h5 (67,108,864행)
[140000/655360] fail: 37385
[160000/655360] fail: 42822
[180000/655360] fail: 48162
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_002.h5 (67,108,864행)
[200000/655360] fail: 53459
[220000/655360] fail: 58826
[240000/655360] fail: 64156
[260000/655360] fail: 69494
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_003.h5 (67,108,864행)
[280000/655360] fail: 74753
[300000/655360] fail: 80139
[320000/655360] fail: 85497
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_004.h5 (67,108,864행)
[340000/655360] fail: 90702
[360000/655360] fail: 95963
[380000/655360] fail: 101212
  청크 저장: /mnt/d/bs_chunks_correction/bs_chunk_005.h5 (67,108,864행)
[400000/655360] fail: 106579
[420000/655360] fail: 111824
[440

In [ ]:
# step1-2 Hes
Hes_paras = generate_hes_valid_params(n_sets=100*(2**16), seed=1234) # 4.1초
hes_eta_dim = Hes_paras.shape[1]

with h5py.File(HES_ETA, "w") as f:
    f.create_dataset("etas", data=Hes_paras,
                    maxshape=(None, hes_eta_dim), 
                    chunks=(10240, hes_eta_dim), 
                    compression="gzip"
                    )

In [ ]:
# step2
chunk_dir  = "/mnt/d/heston_chunks_correction/"
chunk_size = 2**26
generate_dataset(HES_ETA, chunk_dir, 
                 model_type='hes', S0=S0, B=B, 
                 chunk_size=chunk_size
                 ) # 1ROUND : 5시간

In [ ]:
# 4. CVAE
# training
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [256, 256, 128]
batch_size  = 1024 # 4080 : 1024 ~ 2048 / 5090 : 2048 ~ 4096 ~ 
n_epochs    = 200 # loss 수렴할 때까지 
lr          = 1e-3
beta        = 1.0

n_samples = 10000 # n_samples= 1k, 10k, 100k

if model_type == 'hes':
    save_path = 'cvae_heston.pt'
    test_etas = [
        [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5],
        ]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    save_path = 'cvae_bs.pt'
    test_etas = [
        [r, sigma, T],
        ]
    eta_keys  = ['r', 'sigma', 'T']


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type, dim_z, hidden_dims, batch_size, n_epochs, lr, beta, save_path
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


if barr_type == 'barr':
    compare_prices(cvae, B, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples) 
elif barr_type == 'van':
    compare_prices(cvae, None, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples)
time3 = time.time()
print(f"Pricing time: {time3 - time2:.6f}s")

plt.plot(loss_history['recon_loss'], label='Recon')
plt.plot(loss_history['KL_loss'],    label='KL')
plt.plot(loss_history['total_loss'], label='Total')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# inference
trained_model = torch.load(save_path)

cvae = CVAE(
    dim_x       = trained_model['dim_x'],
    dim_eta     = trained_model['dim_eta'],
    dim_z       = trained_model['dim_z'],
    hidden_dims = trained_model['hidden_dims']
)
cvae.load_state_dict(trained_model['model_state'])
cvae.eval()

eta_min = trained_model['eta_min']
eta_max = trained_model['eta_max']

# η 정규화 후 가격 산정
eta_raw    = np.array(test_etas[0])  
eta_scaled = (eta_raw - eta_min) / (eta_max - eta_min + 1e-8)
eta_t      = torch.tensor(eta_scaled, dtype=torch.float32)

if barr_type == 'barr':
    price = cvae.price_barrier(eta_t, B, K, r, T)
elif barr_type == 'van':    
    price = cvae.price_vanilla(eta_t, K, r, T)

In [ ]:
# Total result
#vanilla
print(closed_bs_van, mc_bs_van_gpu, cn_bs_van)
print(f"{(mc_bs_van_gpu - closed_bs_van) / closed_bs_van * 100}%")
print(f"{(cn_bs_van - closed_bs_van) / closed_bs_van * 100}%\n")

print(mc_hes_van_gpu, cs_hes_van)
print(f"{(cs_hes_van - mc_hes_van_gpu) / mc_hes_van_gpu * 100}%\n\n")

#barrier
print(closed_bs_barr, mc_bs_barr_gpu, cn_bs_barr)
print(f"{(mc_bs_barr_gpu - closed_bs_barr) / closed_bs_barr * 100}%")
print(f"{(cn_bs_barr - closed_bs_barr) / closed_bs_barr * 100}%\n")

print(mc_hes_barr_gpu, cs_hes_barr)
print(f"{(cs_hes_barr - mc_hes_barr_gpu) / mc_hes_barr_gpu * 100}%")


In [ ]:
'''
n_list    = [1000, 10000, 100000]
n_repeats = 50
results   = {n: [] for n in n_list}

for n in n_list:
    for _ in range(n_repeats):
        price = MC_heston_barrier_gpu(Hes_eta, B, n_paths=n, type=opt_type) # MC_heston_barrier_gpu
        results[n].append(price)

#MC_BS_vanilla_gpu, MC_heston_vanilla_gpu, MC_BS_barrier_gpu, MC_heston_barrier_gpu
#BS_eta, Hes_eta

# pickle은 binary로.
with open('mc_hes_barr_put_results.pkl', 'wb') as f:
    pickle.dump(results, f)
'''

In [ ]:
'''
configs = [ 
# vanilla
    # call
        dict(dS=0.01, dv=0.01, dt=0.01),  # = 0.12997582
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.12512163, 22s ***
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.12449123, 3m 44s *****
    dict(dS=0.005, dv=0.005, dt=0.005), # = 0.127356
    dict(dS=0.002, dv=0.002, dt=0.005), # = 0.125731, dt=0.01에서 줄여도 큰 차이 없음
    dict(dS=0.002, dv=0.002, dt=0.01), # = 0.125723
    dict(dS=0.002, dv=0.001, dt=0.01), # = 0.125134
    dict(dS=0.001, dv=0.002, dt=0.01), # = 0.125692, 메모리 10% 사용, dv 줄이는게 영향이 더 큼
    dict(dS=0.001, dv=0.001, dt=0.01), # = 0.125106
    dict(dS=0.001, dv=0.0001, dt=0.01), # = 0.124478, 28m, 38% 사용
    dict(dS=0.01, dv=0.00001, dt=0.01), # = 0.124395, 40m 41s
    dict(dS=0.0005, dv=0.0005, dt=0.01), # = 0.124748, 1step 5초
    dict(dS=0.0001, dv=0.0005, dt=0.01), # memory error
    dict(dS=0.0001, dv=0.0001, dt=0.01), # memory error
    
    #put
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.08597330
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.08111911
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.08048871 *****

        
        
 # barr
    # call
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.12014952
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.11625772, 20.3s
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.11573391, 3m 37s ***
    dict(dS=0.01, dv=0.00001, dt=0.01), # = 0.115654, 44m 26s
    dict(dS=0.001, dv=0.0001, dt=0.01), # = 0.115722, 21m 20s
    
    # put
        dict(dS=0.01, dv=0.01, dt=0.01), # = 0.00539354, 3.6s, 
        dict(dS=0.01, dv=0.001, dt=0.01), # = 0.00521346, 20.6s, 
        dict(dS=0.01, dv=0.0001, dt=0.01), # = 0.00517056, 3m 22s,
    dict(dS=0.001, dv=0.01, dt=0.01), # = 0.005410,  
    dict(dS=0.0001, dv=0.01, dt=0.01), # = 0.005410
    dict(dS=0.001, dv=0.001, dt=0.01), # =0.005231
]
'''
'''
configs = [ 
    dict(dS=0.01, dv=0.01, dt=0.01),
    dict(dS=0.01, dv=0.001, dt=0.01),
    dict(dS=0.01, dv=0.0001, dt=0.01),
]

for cfg in configs:
    p = CS_ADI_heston_vanilla(Hes_eta, type=opt_type, **cfg)
    print(cfg, f"→ {p:.8f}")
'''

In [ ]:
fdm_price = 0.124491 # 변수
n_list    = [1000, 10000, 100000]
with open('mc_hes_van_call_results.pkl', 'rb') as f:
    results = pickle.load(f)
    
means  = [np.mean(results[n]) for n in n_list]
stds   = [np.std(results[n])  for n in n_list]
ci     = [1.96 * s for s in stds]

plt.figure(figsize=(7, 4))
plt.axhline(fdm_price, color='gray', linewidth=1.5, label='FDM')
plt.errorbar(range(len(n_list)), means, yerr=ci,
             fmt='o-', color='steelblue', capsize=5, label='MC (95% CI)')
plt.xticks(range(len(n_list)), ['1K', '10K', '100K'])
plt.xlabel('Number of simulations')
plt.ylabel('Option price')
plt.title('convergence')
plt.legend()
plt.tight_layout()